In [1]:
import time
from langgraph.graph import StateGraph,END 
from typing import TypedDict

In [9]:
from langchain_openai import ChatOpenAI 
from langchain_core.tools import tool

In [10]:
### Tool Creation

In [41]:
@tool
def calculator_tool(expression : str) -> str :
    """
    Calculates basic math operations

    Arguments:
        expression -> String type mathematical expression like 25+17 or 200*9

    Returns: expression evaluation     
    """
    print (f'Evaluated expression is {expression}' )
    try : 
        allowed_names = {
            'abs':abs ,
            'round':round,
            'min':min,
            'max':max,
            'sum':sum
        }

        result = eval(expression,
                      {"__builtin__":{}},allowed_names)
        return result
    except : 
        return f'Error calculating : {expression}'

In [13]:
### Define LLM 


llm = ChatOpenAI(
    model = "llama3.2:3b",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    temperature = 0

)

In [14]:
### add tool to llm : 

llm_with_tools = llm.bind_tools([calculator_tool])

In [15]:
### Lang Graph Implementation Starts

In [26]:
### Define State

class State(TypedDict) :
    query : str 
    is_math : bool 
    result : str 




In [27]:
### Classify Query as Math or Text : 
def query_classifier(state : State):
    query_lower = state['query'].lower()
    math_keywords = ['+','-','*','/','divide','add','multiply','sum','calculate','times']
    is_math = any([ word in math_keywords  for word in query_lower])
    return {'is_math':is_math}


In [28]:
### Router Node
def router(state : State):

    if state['is_math'] : 
        return  'calculator'
    return  'general'    

In [43]:
### Define nodes

def calculator_node(state : State):
    prompt = f'Calculate the following expression using calculator_tool {state['query']}'
    response = llm_with_tools.invoke(prompt)
    if hasattr(response,'tool_calls') and response.tool_calls :
        args = response.tool_calls[0].get('args')
        ### Execute Tools
        result = calculator_tool.invoke(args)
        answer = result 
    else :
        answer = response.content.strip()    
    return {'result' : f'Answer : {answer}'}

In [46]:
def general_node (state:State):
    return {'result' : 'Non Math Query'}

In [47]:
###Define LangGraph

In [51]:
workflow = StateGraph(State)

### Add Nodes
workflow.add_node('calculator',calculator_node)
workflow.add_node('general',general_node)
workflow.add_node('classifier',query_classifier)

### Add Start Node
workflow.set_entry_point('classifier')

### Add Edges
workflow.add_conditional_edges(
    'classifier',
    router,
    {
        'calculator' : 'calculator',
        'general' : 'general'
    }
)

workflow.add_edge('calculator' , END)

workflow.add_edge('general' , END)
app = workflow.compile()


In [54]:
test = app.invoke({
    'query' : 'ten plus 10',
    'is_math' : 1,
    'result' : ''
})

In [55]:
print(test)

{'query': 'ten plus 10', 'is_math': False, 'result': 'Non Math Query'}
